# 节点10：GPT-3（2020）——1750 亿参数，少样本学习的涌现

本 notebook 用纯 NumPy 模拟 GPT-3 的核心机制：
- 自回归采样（温度 + top-k）
- In-context learning 格式化
- Scaling Law 可视化
- Mini GPT-3 Block（Pre-LN + Causal Attention）

In [ ]:
import numpy as np
import matplotlib
matplotlib.rcParams["font.family"] = "DejaVu Sans"
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Setup complete")

## Step 1: 自回归采样——温度控制

GPT-3 生成文本时，每次从词汇表的概率分布中**采样**下一个词。

**温度（Temperature）** 控制分布的"尖锐程度"：
- 温度 → 0：总是选概率最高的词（贪心，保守）
- 温度 = 1：按原始概率采样（标准）
- 温度 > 1：分布更平，更随机，更有创意

数学：先把 logits（原始分数）除以温度，再取 softmax：

$$P_T(x_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

In [ ]:
def softmax(x):
    """Numerically stable softmax."""
    x = x - np.max(x)
    e = np.exp(x)
    return e / e.sum()

def sample_with_temperature(logits, temperature=1.0):
    """Sample a token index from logits with temperature scaling."""
    if temperature <= 0:
        return int(np.argmax(logits))
    scaled = logits / temperature
    probs = softmax(scaled)
    return int(np.random.choice(len(probs), p=probs))

# Demonstrate: same logits, different temperatures
vocab = ["apple", "banana", "cherry", "date", "elderberry"]
logits = np.array([3.0, 2.0, 1.0, 0.5, 0.2])  # apple is most likely

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
temps = [0.1, 0.5, 1.0, 2.0]
for ax, T in zip(axes, temps):
    probs = softmax(logits / T) if T > 0 else (np.arange(5) == np.argmax(logits)).astype(float)
    ax.bar(vocab, probs, color="steelblue", alpha=0.7)
    ax.set_title(f"T = {T}")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Temperature Effect on Sampling Distribution", y=1.02)
plt.tight_layout()
plt.savefig("../docs/assets/10_temperature.png", dpi=80, bbox_inches="tight")
plt.show()
print("T=0.1 → near-greedy, T=2.0 → diverse sampling")

## Step 2: Top-k 采样

Temperature 采样的问题：即使概率极低的词也可能被采到，生成胡言乱语。

**Top-k 采样**：只保留概率最高的 k 个词，其余置零再重新归一化。

这样既有随机性（有趣），又限制了"踩雷"（不会生成完全不合理的词）。

In [ ]:
def top_k_sample(logits, k=5, temperature=1.0):
    """Top-k sampling: keep only top-k logits, then sample."""
    scaled = logits / max(temperature, 1e-8)
    # Get top-k indices
    top_k_idx = np.argsort(scaled)[-k:]
    # Mask out non-top-k
    masked = np.full_like(scaled, -np.inf)
    masked[top_k_idx] = scaled[top_k_idx]
    probs = softmax(masked)
    return int(np.random.choice(len(probs), p=probs))

# Simulate a short autoregressive generation loop
VOCAB_SIZE = 20
SEQ_LEN = 10

def autoregressive_generate(seed_tokens, n_new=5, k=5, temperature=0.8):
    """Simple autoregressive generation using random "model" logits."""
    tokens = list(seed_tokens)
    for step in range(n_new):
        # In a real GPT, the model would compute logits from all past tokens
        # Here we simulate with deterministic pseudo-logits based on position
        np.random.seed(step + sum(tokens))  # reproducible but position-dependent
        logits = np.random.randn(VOCAB_SIZE)
        next_tok = top_k_sample(logits, k=k, temperature=temperature)
        tokens.append(next_tok)
    return tokens

seed = [1, 5, 3]
generated = autoregressive_generate(seed, n_new=7, k=5, temperature=0.8)
print(f"Seed tokens:      {seed}")
print(f"Generated tokens: {generated}")
print(f"New tokens added: {generated[len(seed):]}")

## Step 3: In-Context Learning——格式的魔力

GPT-3 最核心的发现：**只要把例子放进提示词，模型就能"学会"新任务**，无需更新任何参数。

下面我们模拟三种 ICL 格式，观察语言模型在生成时如何利用上下文。

关键洞察：模型在预训练时看过无数"例子→答案"格式的文本，所以它"知道"这种格式意味着什么。

In [ ]:
def format_zero_shot(task_description, query):
    """Zero-shot: just the task description + query."""
    return f"{task_description}\nInput: {query}\nOutput:"

def format_one_shot(task_description, example_in, example_out, query):
    """One-shot: one example + query."""
    return (
        f"{task_description}\n"
        f"Input: {example_in}\nOutput: {example_out}\n\n"
        f"Input: {query}\nOutput:"
    )

def format_few_shot(task_description, examples, query):
    """Few-shot: multiple examples + query."""
    prompt = task_description + "\n"
    for inp, out in examples:
        prompt += f"Input: {inp}\nOutput: {out}\n\n"
    prompt += f"Input: {query}\nOutput:"
    return prompt

# Sentiment analysis examples
task = "Classify sentiment as Positive or Negative."
examples = [
    ("This movie is amazing!", "Positive"),
    ("I hate waiting in long lines.", "Negative"),
    ("The food was absolutely delicious.", "Positive"),
]
query = "The service was terrible and the room was dirty."

print("=== Zero-shot prompt ===")
print(format_zero_shot(task, query))
print()
print("=== One-shot prompt ===")
print(format_one_shot(task, examples[0][0], examples[0][1], query))
print()
print("=== Few-shot prompt (3 examples) ===")
print(format_few_shot(task, examples, query))

## Step 4: 规模律（Scaling Laws）可视化

Kaplan et al. (2020) 发现，语言模型的损失和参数量之间满足**幂律关系**：

$$L(N) \approx \left(\frac{N_c}{N}\right)^{\alpha_N}$$

在 log-log 坐标下，这是一条直线。GPT-3 的设计就是沿着这条线外推的。

下面我们可视化实际的规模律数据（来自 Kaplan et al. 论文的近似值）。

In [ ]:
# Approximate data from Kaplan et al. 2020 (Figure 1)
# (N, L) pairs: parameter count vs validation loss
scaling_data = [
    (1e6,   4.60),   # 1M params
    (3e6,   4.00),
    (1e7,   3.50),
    (3e7,   3.10),
    (1e8,   2.80),
    (3e8,   2.55),
    (1e9,   2.35),
    (3e9,   2.18),
    (1e10,  2.05),
    (3e10,  1.92),
    (1e11,  1.82),   # ~100B
    (1.75e11, 1.77), # GPT-3 (175B)
]

params = np.array([d[0] for d in scaling_data])
losses = np.array([d[1] for d in scaling_data])

# Fit power law: log(L) = alpha * log(N) + C
log_params = np.log10(params)
log_losses = np.log10(losses)
coeffs = np.polyfit(log_params, log_losses, 1)
alpha = coeffs[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Linear scale
ax1.scatter(params, losses, color="steelblue", s=50, zorder=3)
ax1.scatter([1.75e11], [1.77], color="red", s=120, zorder=4, label="GPT-3 (175B)")
ax1.set_xscale("log")
ax1.set_xlabel("Parameters (log scale)")
ax1.set_ylabel("Validation Loss")
ax1.set_title("Scaling Law: Parameters vs Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Log-log scale (should be linear)
fit_params = np.logspace(6, 11.5, 100)
fit_losses = 10 ** np.polyval(coeffs, np.log10(fit_params))
ax2.scatter(log_params, log_losses, color="steelblue", s=50, zorder=3)
ax2.scatter([np.log10(1.75e11)], [np.log10(1.77)], color="red", s=120, zorder=4, label="GPT-3 (175B)")
ax2.plot(np.log10(fit_params), np.log10(fit_losses), "k--", alpha=0.7, label=f"Power law fit (α={alpha:.3f})")
ax2.set_xlabel("log₁₀(Parameters)")
ax2.set_ylabel("log₁₀(Loss)")
ax2.set_title("Log-Log: Linear Relationship")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../docs/assets/10_scaling_law.png", dpi=80, bbox_inches="tight")
plt.show()
print(f"Power law exponent α ≈ {alpha:.3f}")
print(f"Interpretation: 10x more params → loss multiplied by 10^{alpha:.3f} = {10**alpha:.3f}")

## Step 5: Mini GPT Block（Pre-LN Transformer Decoder）

GPT-3 的网络架构和 GPT-2 几乎相同（Pre-LN + Causal Attention + FFN）。
区别只在于规模：GPT-3 有 96 层，每层隐向量维度 12288，96 个注意力头。

下面实现一个完整的 Mini GPT Block，验证前向传播的形状和行为。

In [ ]:
def layer_norm(x, eps=1e-5):
    """Layer normalization."""
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

def gelu(x):
    """GELU activation (approximate)."""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

def causal_attention(Q, K, V, mask):
    """Scaled dot-product attention with causal mask."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    scores = np.where(mask, scores, -1e9)
    # Apply softmax row-wise
    attn_rows = np.stack([softmax(row) for row in scores])
    return attn_rows @ V

def mini_gpt_block(x, d_model=32, d_ff=64):
    """One Pre-LN GPT block: LN → Attention → residual → LN → FFN → residual."""
    seq_len = x.shape[0]
    mask = np.tril(np.ones((seq_len, seq_len), dtype=bool))

    # Pre-LN + Self-Attention
    x_norm = layer_norm(x)
    np.random.seed(0)
    Wq = np.random.randn(d_model, d_model) * 0.02
    Wk = np.random.randn(d_model, d_model) * 0.02
    Wv = np.random.randn(d_model, d_model) * 0.02
    Q = x_norm @ Wq
    K = x_norm @ Wk
    V = x_norm @ Wv
    attn_out = causal_attention(Q, K, V, mask)
    x = x + attn_out  # residual

    # Pre-LN + FFN
    x_norm2 = layer_norm(x)
    W1 = np.random.randn(d_model, d_ff) * 0.02
    W2 = np.random.randn(d_ff, d_model) * 0.02
    ffn_out = gelu(x_norm2 @ W1) @ W2
    x = x + ffn_out  # residual

    return x

# Test with a sequence of 8 tokens, d_model=32
seq_len, d_model = 8, 32
x = np.random.randn(seq_len, d_model).astype(np.float32)
out = mini_gpt_block(x, d_model=d_model, d_ff=64)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Shape preserved: {x.shape == out.shape}")
print(f"Output norm (mean): {np.linalg.norm(out, axis=-1).mean():.3f}")


## Step 6: 涌现能力模拟——算术任务

GPT-3 论文中一个令人惊讶的发现：模型能做两位数加法（~98% 准确率）。
这能力在小模型中几乎不存在，在大模型中"突然出现"——这就是**涌现（emergence）**。

下面我们模拟 few-shot 算术的 prompt 格式，以及简单的词元级加法。

In [ ]:
def format_arithmetic_few_shot(examples, query):
    """Format arithmetic few-shot prompt."""
    prompt = "Solve the arithmetic problem.\n\n"
    for a, b, ans in examples:
        prompt += f"Q: {a} + {b} = ?\nA: {ans}\n\n"
    prompt += f"Q: {query[0]} + {query[1]} = ?\nA:"
    return prompt

examples = [(12, 34, 46), (55, 23, 78), (71, 18, 89)]
query = (47, 35)
prompt = format_arithmetic_few_shot(examples, query)
print(prompt)
print(f"\nExpected answer: {query[0] + query[1]}")

# Simulate "emergent" accuracy vs model size (from Kaplan 2020 / GPT-3 paper)
model_sizes = [125e6, 350e6, 1.3e9, 6.7e9, 13e9, 175e9]
two_digit_acc = [0.02, 0.05, 0.15, 0.45, 0.75, 0.98]  # approximate

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([np.log10(s) for s in model_sizes], two_digit_acc, "o-", color="steelblue", linewidth=2)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="50% threshold")
ax.set_xlabel("log₁₀(Parameters)")
ax.set_ylabel("Accuracy on 2-digit addition")
ax.set_title("Emergent Arithmetic: Accuracy vs Model Size")
ax.set_xticks([np.log10(s) for s in model_sizes])
ax.set_xticklabels(["125M", "350M", "1.3B", "6.7B", "13B", "175B"])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../docs/assets/10_emergence.png", dpi=80, bbox_inches="tight")
plt.show()
print("Emergent capability: near-zero at small scale, ~98% at GPT-3 scale")

## 总结

| 概念 | 核心公式/机制 | 一句话 |
|------|-------------|--------|
| 自回归采样 | $P_T(x_i) = \text{softmax}(z_i/T)$ | 温度越低越保守，越高越随机 |
| Top-k 采样 | 只保留前 k 个 logit | 随机但不胡言乱语 |
| In-context learning | prompt 里放例子，参数不变 | 不训练，靠格式"教"模型 |
| 规模律 | $L(N) \approx (N_c/N)^{\alpha}$ | log-log 下是直线，可外推 |
| 涌现能力 | 小模型几乎 0，大模型突然出现 | 量变引发质变 |

**GPT-3 的历史意义**：证明了"大就是强"——规模本身就是一种算法。
它留下的问题（不听指令、幻觉、成本高）直接催生了 InstructGPT 和 ChatGPT。